# PointNet part segmentation, scored by IoU

A point cloud is an unordered set of coordinates. Convolution assumes a grid, so it does
not apply; PointNet handles the set directly by learning a per-point embedding and
pooling it symmetrically, which makes the output invariant to the order the points arrive
in. This notebook trains that architecture **from scratch** - no pretrained weights - to
label which part of an object each point belongs to: which points of a chair are its
legs, which are its seat.

The dataset is ShapeNetPart in its canonical HDF5 packaging: 2048 sampled points per
shape, 16 object categories, 50 part labels, with the official train / val / test files
kept separate. Scoring happens on the **test** files only, which the model never sees
during training or model selection.

The metric is intersection-over-union, reported three ways, because a single averaged
number hides most of what you want to know:

- **instance mIoU** - per shape, average IoU over the parts of its own category, then
  average over shapes. This is the number the PointNet paper reports.
- **class mIoU** - average the per-category scores, so a category with 30 examples counts
  as much as one with 3000.
- **per-category IoU** - the full table, because the average is a poor summary of a
  distribution that ranges from nearly-solved to nearly-failed.

One architecture on one dataset establishes what this architecture scores here. It does
not establish that it beats any other method; no comparison is run, so none is claimed.


## What was actually attached

Before assuming a directory layout, look.

In [ ]:
import os, sys, glob, json

# Walk the attached dataset before assuming anything about its shape. A wrong path
# assumption costs a whole GPU session, so look first and print what is actually there.
layout = {}
for root in ("/kaggle/input",):
    if not os.path.isdir(root):
        continue
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) > 3:
            dirnames[:] = []
            continue
        exts = {}
        for f in filenames:
            e = os.path.splitext(f)[1].lower()
            exts[e] = exts.get(e, 0) + 1
        if exts:
            layout[dirpath] = exts

print(f"{len(layout)} directories with files\n")
for d, exts in sorted(layout.items())[:40]:
    top = ", ".join(f"{k or '(none)'}:{v}" for k, v in
                    sorted(exts.items(), key=lambda kv: -kv[1])[:6])
    print(f"  {d}\n      {top}")

H5 = sorted(glob.glob("/kaggle/input/**/*.h5", recursive=True))
print(f"\n{len(H5)} HDF5 files found")
for p in H5:
    print("   ", p.replace("/kaggle/input/", ""))

## The official splits, and the keys inside the HDF5 files

In [ ]:
import h5py, numpy as np

def bucket(paths):
    """Group HDF5 files into the official splits by filename, not by directory depth."""
    out = {"train": [], "val": [], "test": []}
    for p in paths:
        b = os.path.basename(p).lower()
        if b.startswith("train") or "_train" in b:
            out["train"].append(p)
        elif b.startswith("val") or "_val" in b:
            out["val"].append(p)
        elif b.startswith("test") or "_test" in b:
            out["test"].append(p)
    return out

SPLITS = bucket(H5)
for k, v in SPLITS.items():
    print(f"{k:6} {len(v)} file(s)")

assert SPLITS["train"] and SPLITS["test"], "need at least a train and a test file"

# Discover the key names by inspecting ONE file. ShapeNetPart HDF5 is usually
# data/label/pid, but the names vary by re-upload, so identify them by shape:
#   (N, P, 3) -> coordinates;  (N, 1) or (N,) -> object category;  (N, P) -> part label.
with h5py.File(SPLITS["train"][0], "r") as f:
    print("\nkeys in", os.path.basename(SPLITS["train"][0]))
    shapes = {}
    for k in f.keys():
        shapes[k] = f[k].shape
        print(f"   {k:12} {f[k].shape}  {f[k].dtype}")

K_XYZ = next(k for k, s in shapes.items() if len(s) == 3 and s[2] >= 3)
K_SEG = next(k for k, s in shapes.items() if len(s) == 2 and s[1] > 3)
K_CAT = next(k for k, s in shapes.items()
             if (len(s) == 2 and s[1] == 1) or len(s) == 1)
N_POINTS = shapes[K_XYZ][1]
print(f"\nusing: xyz={K_XYZ}  part_label={K_SEG}  category={K_CAT}  "
      f"points_per_shape={N_POINTS}")

## Loading, and a check that the part numbering is the standard one

In [ ]:
CAT_PARTS = json.loads(r'''{"Airplane": [0, 1, 2, 3], "Bag": [4, 5], "Cap": [6, 7], "Car": [8, 9, 10, 11], "Chair": [12, 13, 14, 15], "Earphone": [16, 17, 18], "Guitar": [19, 20, 21], "Knife": [22, 23], "Lamp": [24, 25, 26, 27], "Laptop": [28, 29], "Motorbike": [30, 31, 32, 33, 34, 35], "Mug": [36, 37], "Pistol": [38, 39, 40], "Rocket": [41, 42, 43], "Skateboard": [44, 45, 46], "Table": [47, 48, 49]}''')
CATS = list(CAT_PARTS.keys())
N_CAT, N_PART = len(CATS), sum(len(v) for v in CAT_PARTS.values())
PART_OF_CAT = np.zeros((N_CAT, N_PART), dtype=bool)
for ci, c in enumerate(CATS):
    PART_OF_CAT[ci, CAT_PARTS[c]] = True
print(f"{N_CAT} categories, {N_PART} part labels")

def load_split(paths):
    X, Y, C = [], [], []
    for p in paths:
        with h5py.File(p, "r") as f:
            X.append(np.asarray(f[K_XYZ][:, :, :3], dtype="float32"))
            Y.append(np.asarray(f[K_SEG][:], dtype="int64"))
            C.append(np.asarray(f[K_CAT][:], dtype="int64").reshape(-1))
    return np.concatenate(X), np.concatenate(Y), np.concatenate(C)

Xtr, Ytr, Ctr = load_split(SPLITS["train"])
Xte, Yte, Cte = load_split(SPLITS["test"])
if SPLITS["val"]:
    Xva, Yva, Cva = load_split(SPLITS["val"])
else:
    # No official val file in this upload: carve one off the END of train, so the
    # test files stay untouched. Model selection must never see test.
    n = len(Xtr) // 10
    Xva, Yva, Cva = Xtr[-n:], Ytr[-n:], Ctr[-n:]
    Xtr, Ytr, Ctr = Xtr[:-n], Ytr[:-n], Ctr[:-n]
    print(f"no val file found; held out the last {n} train shapes for validation")

for nm, (X, Y, C) in (("train", (Xtr, Ytr, Ctr)), ("val", (Xva, Yva, Cva)),
                      ("test", (Xte, Yte, Cte))):
    print(f"{nm:6} {len(X):6d} shapes   part labels {Y.min()}..{Y.max()}   "
          f"categories {C.min()}..{C.max()}")

# Sanity: do the part labels present in each category match the standard ranges? If a
# re-upload numbered them differently every per-category IoU below would be wrong.
bad = 0
for ci in range(N_CAT):
    m = Ctr == ci
    if not m.any():
        continue
    seen = set(np.unique(Ytr[m]).tolist())
    if not seen.issubset(set(CAT_PARTS[CATS[ci]])):
        bad += 1
        print(f"  MISMATCH {CATS[ci]}: saw {sorted(seen)} expected {CAT_PARTS[CATS[ci]]}")
print("label-range check:", "all 16 categories match the standard numbering" if bad == 0
      else f"{bad} categories disagree - per-category IoU would be misassigned")
# Fail fast rather than report a mislabelled per-category table. Per-category IoU is a
# required output here, and a silently misassigned one is worse than no result at all.
assert bad == 0, (
    f"{bad} categories do not match the standard ShapeNetPart numbering, so the "
    f"per-category IoU table would attribute parts to the wrong categories. "
    f"Inspect the MISMATCH lines above before trusting any number from this run.")

# The HDF5 key names in this packaging are data/label/seg, NOT the data/label/pid that
# most ShapeNetPart code assumes - which is why the keys above are discovered by array
# SHAPE rather than by name. Verified against the real test1.h5 header.
print(f"keys resolved by shape: xyz={K_XYZ}, part={K_SEG}, category={K_CAT}")

## PointNet

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

class TNet(nn.Module):
    """Predicts a KxK alignment matrix. Lets the network canonicalise its own input
    orientation instead of relying on the dataset to be pre-aligned."""
    def __init__(self, k):
        super().__init__()
        self.k = k
        self.conv = nn.Sequential(
            nn.Conv1d(k, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU())
        self.fc = nn.Sequential(
            nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, k * k))
        nn.init.zeros_(self.fc[-1].weight); nn.init.zeros_(self.fc[-1].bias)

    def forward(self, x):
        b = x.shape[0]
        h = self.conv(x).max(dim=2).values
        m = self.fc(h).view(b, self.k, self.k)
        return m + torch.eye(self.k, device=x.device).unsqueeze(0)

class PointNetSeg(nn.Module):
    """PointNet segmentation head. The max-pool is what makes it permutation invariant;
    the per-point features are concatenated back so each point keeps its local identity
    alongside the global shape descriptor. The one-hot category code is appended because
    part labels are only meaningful within a category."""
    def __init__(self, n_part, n_cat):
        super().__init__()
        self.tn1, self.tn2 = TNet(3), TNet(64)
        self.c1 = nn.Sequential(nn.Conv1d(3, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
                                nn.Conv1d(64, 64, 1), nn.BatchNorm1d(64), nn.ReLU())
        self.c2 = nn.Sequential(nn.Conv1d(64, 64, 1), nn.BatchNorm1d(64), nn.ReLU(),
                                nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
                                nn.Conv1d(128, 1024, 1), nn.BatchNorm1d(1024), nn.ReLU())
        self.head = nn.Sequential(
            nn.Conv1d(1024 + 64 + 64 + 128 + n_cat, 512, 1), nn.BatchNorm1d(512), nn.ReLU(),
            nn.Conv1d(512, 256, 1), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.2),
            nn.Conv1d(256, 128, 1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.Conv1d(128, n_part, 1))
        self.n_cat = n_cat

    def forward(self, x, cat):
        b, _, n = x.shape
        x = torch.bmm(self.tn1(x).transpose(1, 2), x)
        f1 = self.c1(x)
        t2 = self.tn2(f1)
        f1 = torch.bmm(t2.transpose(1, 2), f1)
        f2 = self.c2[0:3](f1)
        f3 = self.c2[3:6](f2)
        f4 = self.c2[6:9](f3)
        g = f4.max(dim=2, keepdim=True).values.repeat(1, 1, n)
        oh = F.one_hot(cat, self.n_cat).float().unsqueeze(2).repeat(1, 1, n)
        return self.head(torch.cat([g, f1, f2, f3, oh], 1)), t2

def ortho_penalty(m):
    """The feature transform is unconstrained and can collapse; the paper regularises it
    toward orthogonality. Without this the 64x64 T-Net destabilises training."""
    i = torch.eye(m.shape[1], device=m.device).unsqueeze(0)
    return ((torch.bmm(m, m.transpose(1, 2)) - i) ** 2).sum(dim=(1, 2)).mean()

model = PointNetSeg(N_PART, N_CAT).to(DEV)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")

## Training, with IoU measured the way the paper measures it

In [ ]:
from torch.utils.data import Dataset, DataLoader
import time

class PC(Dataset):
    def __init__(self, X, Y, C, aug):
        self.X, self.Y, self.C, self.aug = X, Y, C, aug
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].copy()
        # Normalise to the unit sphere: scale is not a part cue and varies by category.
        x -= x.mean(0); x /= (np.linalg.norm(x, axis=1).max() + 1e-9)
        if self.aug:
            a = np.random.uniform(0, 2 * np.pi)                 # yaw only: these shapes
            ca, sa = np.cos(a), np.sin(a)                       # are gravity-aligned, so
            R = np.array([[ca, 0, sa], [0, 1, 0], [-sa, 0, ca]], "float32")
            x = x @ R
            x *= np.random.uniform(0.8, 1.25)
            x += np.clip(np.random.normal(0, 0.01, x.shape), -0.05, 0.05).astype("float32")
        return (torch.from_numpy(x.astype("float32").T),
                torch.tensor(int(self.C[i])), torch.from_numpy(self.Y[i]))

BS, EPOCHS = 32, 40
dl_tr = DataLoader(PC(Xtr, Ytr, Ctr, True), BS, shuffle=True, num_workers=2,
                   drop_last=True, pin_memory=True)
dl_va = DataLoader(PC(Xva, Yva, Cva, False), 64, num_workers=2)
dl_te = DataLoader(PC(Xte, Yte, Cte, False), 64, num_workers=2)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS, eta_min=1e-5)
scaler = torch.cuda.amp.GradScaler()
MASK = torch.from_numpy(PART_OF_CAT).to(DEV)

@torch.no_grad()
def evaluate(dl):
    """Instance IoU, per the PointNet protocol: for each shape, average IoU over the
    parts of ITS category only. A part absent from both prediction and truth counts as
    1.0 - it is correctly absent - which is the convention the published numbers use.
    Predictions are restricted to the shape's own category, as the paper does."""
    model.eval()
    per_shape, per_cat = [], {c: [] for c in range(N_CAT)}
    conf = np.zeros((N_PART, 3), dtype="int64")      # inter, pred, true  (for global IoU)
    for x, c, y in dl:
        x, c, y = x.to(DEV), c.to(DEV), y.to(DEV)
        with torch.cuda.amp.autocast():
            logits, _ = model(x, c)
        logits = logits.float()
        logits = logits.masked_fill(~MASK[c].unsqueeze(2), -1e4)
        pred = logits.argmax(1)
        for i in range(x.shape[0]):
            ci = int(c[i]); ious = []
            for p in CAT_PARTS[CATS[ci]]:
                pi, ti = pred[i] == p, y[i] == p
                inter = (pi & ti).sum().item()
                union = (pi | ti).sum().item()
                ious.append(1.0 if union == 0 else inter / union)
                conf[p] += (inter, pi.sum().item(), ti.sum().item())
            m = float(np.mean(ious))
            per_shape.append(m); per_cat[ci].append(m)
    global_iou = [conf[p, 0] / max(conf[p, 1] + conf[p, 2] - conf[p, 0], 1)
                  for p in range(N_PART) if conf[p, 2] > 0]
    return {"instance_miou": float(np.mean(per_shape)),
            "class_miou": float(np.mean([np.mean(v) for v in per_cat.values() if v])),
            "global_part_miou": float(np.mean(global_iou)),
            "per_category": {CATS[k]: float(np.mean(v)) for k, v in per_cat.items() if v},
            "per_category_n": {CATS[k]: len(v) for k, v in per_cat.items() if v},
            "n_shapes": len(per_shape)}

hist, best, best_state = [], -1, None
print(f"training {EPOCHS} epochs on {len(Xtr)} shapes")
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = nb = 0
    for x, c, y in dl_tr:
        x, c, y = x.to(DEV, non_blocking=True), c.to(DEV), y.to(DEV)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            logits, t2 = model(x, c)
            loss = F.cross_entropy(logits, y) + 1e-3 * ortho_penalty(t2.float())
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += loss.item(); nb += 1
    sched.step()
    row = {"epoch": ep + 1, "train_loss": tot / nb, "lr": opt.param_groups[0]["lr"]}
    if (ep + 1) % 4 == 0 or ep == EPOCHS - 1:
        v = evaluate(dl_va)
        row["val_instance_miou"] = v["instance_miou"]
        row["val_class_miou"] = v["class_miou"]
        if v["instance_miou"] > best:
            best = v["instance_miou"]
            best_state = {k: p.detach().clone() for k, p in model.state_dict().items()}
            row["best"] = True
    hist.append(row)
    print(f"  ep {ep + 1:3d}  loss {row['train_loss']:.4f}  "
          f"val_inst_mIoU {row.get('val_instance_miou', float('nan')):.4f}  "
          f"{time.time() - t0:.0f}s" + ("  <- best" if row.get("best") else ""))

model.load_state_dict(best_state)
print(f"\nrestored the epoch with the best VALIDATION mIoU ({best:.4f}); "
      f"test has not been touched yet")

## Result on the held-out test split

In [ ]:
res_tr = evaluate(DataLoader(PC(Xtr, Ytr, Ctr, False), 64, num_workers=2))
res_va = evaluate(dl_va)
res_te = evaluate(dl_te)

print(f"{'split':8}{'n':>7}{'instance mIoU':>16}{'class mIoU':>13}{'global part mIoU':>19}")
for nm, r in (("train", res_tr), ("val", res_va), ("test", res_te)):
    print(f"{nm:8}{r['n_shapes']:>7}{r['instance_miou']:>16.4f}"
          f"{r['class_miou']:>13.4f}{r['global_part_miou']:>19.4f}")

print(f"\nper-category IoU on the held-out TEST split")
print(f"{'category':14}{'n':>6}{'IoU':>9}   {'train':>7}")
for c in sorted(res_te["per_category"], key=lambda k: -res_te["per_category"][k]):
    print(f"{c:14}{res_te['per_category_n'][c]:>6}{res_te['per_category'][c]:>9.4f}"
          f"   {res_tr['per_category'].get(c, float('nan')):>7.4f}")

gap = res_tr["instance_miou"] - res_te["instance_miou"]
print(f"\ntrain - test instance mIoU gap: {gap:+.4f}")

results = {"task": "ShapeNetPart part segmentation",
           "model": "PointNet (segmentation head, input + feature T-Net), from scratch",
           "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 3),
           "points_per_shape": int(N_POINTS), "epochs": EPOCHS, "batch_size": BS,
           "n_categories": N_CAT, "n_parts": N_PART,
           "selection": "epoch chosen by validation instance mIoU; test untouched",
           "train": res_tr, "val": res_va, "test": res_te,
           "train_test_gap_instance_miou": round(gap, 4),
           "history": hist}
with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=1)
print("\nwrote /kaggle/working/results.json")
print(json.dumps({k: results[k] for k in
                  ("task", "model", "points_per_shape", "epochs")}, indent=1))

## Curves, and the per-category spread

In [ ]:
import matplotlib.pyplot as plt

ep = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), dpi=130)
ax[0].plot(ep, [h["train_loss"] for h in hist], lw=1.6)
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("train loss"); ax[0].set_title(
    "training loss", fontsize=10)
v = [(h["epoch"], h["val_instance_miou"]) for h in hist if "val_instance_miou" in h]
ax[1].plot([a for a, _ in v], [b for _, b in v], "o-", lw=1.6, ms=3.5, label="val")
ax[1].axhline(res_te["instance_miou"], ls="--", c="crimson", lw=1.3,
              label=f"test {res_te['instance_miou']:.3f}")
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("instance mIoU")
ax[1].set_title("validation mIoU, with the final test score", fontsize=10)
ax[1].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.25, lw=0.5); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

# Per-category, sorted. The average hides this spread, which is the point of showing it.
cats = sorted(res_te["per_category"], key=lambda k: res_te["per_category"][k])
plt.figure(figsize=(7.5, 4.4), dpi=130)
plt.barh(cats, [res_te["per_category"][c] for c in cats], color="#3b6ea5", height=0.68)
plt.axvline(res_te["class_miou"], ls="--", c="crimson", lw=1.2,
            label=f"class mIoU {res_te['class_miou']:.3f}")
for i, c in enumerate(cats):
    plt.text(res_te["per_category"][c] + 0.008, i,
             f"{res_te['per_category'][c]:.3f}  (n={res_te['per_category_n'][c]})",
             va="center", fontsize=7.5)
plt.xlim(0, 1.18); plt.xlabel("IoU on held-out test shapes")
plt.title("per-category part IoU", fontsize=10); plt.legend(fontsize=8, loc="lower right")
plt.gca().spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## Where it fails

In [ ]:
# Show predictions where they are wrong, not only where they are right: the four
# test shapes with the LOWEST IoU alongside four near the median.
model.eval()
scored = []
with torch.no_grad():
    for x, c, y in DataLoader(PC(Xte, Yte, Cte, False), 64, num_workers=2):
        xd, cd, yd = x.to(DEV), c.to(DEV), y.to(DEV)
        with torch.cuda.amp.autocast():
            lg, _ = model(xd, cd)
        lg = lg.float().masked_fill(~MASK[cd].unsqueeze(2), -1e4)
        pr = lg.argmax(1)
        for i in range(x.shape[0]):
            ci = int(c[i])
            ious = []
            for p in CAT_PARTS[CATS[ci]]:
                pi, ti = pr[i] == p, yd[i] == p
                u = (pi | ti).sum().item()
                ious.append(1.0 if u == 0 else (pi & ti).sum().item() / u)
            scored.append((float(np.mean(ious)), x[i].numpy().T,
                           yd[i].cpu().numpy(), pr[i].cpu().numpy(), CATS[ci]))
scored.sort(key=lambda t: t[0])
mid = len(scored) // 2
picks = scored[:4] + scored[mid:mid + 4]

fig = plt.figure(figsize=(13, 7.0), dpi=130)
for j, (iou, pts, gt, pd_, cname) in enumerate(picks):
    for k, (lab, ttl) in enumerate(((gt, "ground truth"), (pd_, "predicted"))):
        ax = fig.add_subplot(4, 4, (j // 4) * 8 + (j % 4) * 2 + k + 1, projection="3d")
        ax.scatter(pts[:, 0], pts[:, 2], pts[:, 1], c=lab, cmap="tab10", s=1.1,
                   linewidths=0, vmin=0, vmax=N_PART - 1)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([]); ax.set_box_aspect((1, 1, 1))
        ax.set_title(f"{cname} {ttl}" if k == 0 else f"IoU {iou:.2f}  {ttl}", fontsize=7.5)
        ax.view_init(elev=18, azim=-62)
fig.suptitle("top row pair: the 4 worst test shapes.  bottom: 4 at the median.",
             fontsize=9, y=0.995)
plt.tight_layout(); plt.show()

## What this shows, and what it does not

PointNet, trained from scratch on ShapeNetPart, reaches the instance mIoU printed above
on the official **test** files. The epoch was chosen on a validation split; the test files
were held out of both training and selection, so the number is not a best-of-many pick.

What it does **not** establish:

- **No comparison is run.** One architecture on one dataset says what this architecture
  scores here. It says nothing about PointNet++, DGCNN, point transformers, or any sparse
  convolutional method, because none of them were trained here. A claim about which is
  better would need all of them trained under the same budget on the same split.
- **Part IoU is not shape understanding.** The category label is given to the model as a
  one-hot input, exactly as in the published protocol. The model decides which part a
  point belongs to; it never has to decide what the object is.
- **The per-category spread is the real result.** Several categories have fewer than 100
  test shapes, so their IoU carries a wide error bar that a single mIoU conceals. The
  per-category table is printed for that reason.
- **2048 sampled points is not the mesh.** These clouds are a uniform resampling of the
  original surfaces; thin structures that fall between samples are not in the input at
  all, and no method can recover them from this representation.

Part of a portfolio: [github.com/hammasbuilds](https://github.com/hammasbuilds) ·
[vision-portfolio-woad.vercel.app](https://vision-portfolio-woad.vercel.app)
